# 宇宙の膨張速度 H(z) を 3 つの方法で測って比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/cosmology/hz_comparison.ipynb)

宇宙の膨張の速さは、ハッブルパラメータ $H(z) = \dot{a}/a$ で表されます。今の値が $H_0$(ハッブル定数)です。
[sn_cosmology_fit.ipynb](sn_cosmology_fit.ipynb) では、超新星だけからモデルを仮定して $H(z)$ を計算しました。

このノートブックでは、**仕組みのまったく違う 3 つの方法** で $H(z)$ を測り、1 枚のグラフに重ねて比べます。

| 方法 | 何を測るか | 物差し(目盛りの基準) |
| --- | --- | --- |
| **宇宙クロノメーター** | 年齢差の分かる銀河から $dz/dt$ を測り、$H(z)$ を直接出す | 恒星の進化モデル(距離はしごは使わない) |
| **BAO**(バリオン音響振動) | 銀河の分布に刻まれた「音波の跡」の見かけの大きさから $H(z)\,r_d$ を出す | 初期宇宙の音波が進んだ距離 $r_d$(CMB から計算) |
| **Ia 型超新星** | 明るさから距離を出し、距離の増え方から $H(z)/H_0$ を出す | セファイド(距離はしご)で決めた $H_0$ |

目標は次の 2 点です。

1. 膨張の歴史の **形**($H(z)/H_0$)は 3 つの方法で合うか
2. 膨張の速さの **絶対値**($H_0$)は合うか(ハッブル・テンション)

## 使うデータ

- **宇宙クロノメーター**: Moresco et al. (2012, 2015, 2016) の $H(z)$ と、Moresco et al. (2020) の系統誤差の見積もり
  https://gitlab.com/mmoresco/CCcovariance
- **BAO**: DESI DR2(DESI Collaboration 2025)の測定値と共分散行列。Cobaya が配布しているもの
  https://github.com/CobayaSampler/bao_data
- **Ia 型超新星**: Pantheon+(Scolnic et al. 2022, Brout et al. 2022)
  https://github.com/PantheonPlusSH0ES/DataRelease

いずれもリポジトリにライセンスの明記はありません。研究用に公開されたデータなので、使うときは論文を引用します。

## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.stats import chi2 as chi2_dist

CC_URL = "https://gitlab.com/mmoresco/CCcovariance/-/raw/master/data/"
BAO_URL = "https://raw.githubusercontent.com/CobayaSampler/bao_data/master/desi_bao_dr2/"
SN_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
)
C_KMS = 299_792.458  # 光速 [km/s]

# 比較に使う値
H0_SH0ES, H0_SH0ES_ERR = 73.4, 1.0  # sn_cosmology_fit.ipynb 6 章の値(SH0ES の論文値は 73.0 ± 1.0)
PLANCK = dict(H0=67.4, H0_err=0.5, Om=0.315, Om_err=0.007)  # Planck 2018 (TT,TE,EE+lowE+lensing)
RD_PLANCK, RD_PLANCK_ERR = 147.09, 0.26  # 音響地平線 r_d [Mpc], Planck 2018

LEVELS_2D = chi2_dist.ppf([0.683, 0.954], df=2)  # 2 パラメータの 1σ, 2σ


def e_lcdm(z, om):
    # 平坦 ΛCDM の E(z) = H(z)/H0
    return np.sqrt(om * (1 + z) ** 3 + 1 - om)


def marginal(values, p, axis):
    # 2 次元の確率 p を 1 軸に周辺化して、平均と標準偏差を返す
    q = p.sum(axis=axis)
    q = q / q.sum()
    mean = np.sum(q * values)
    return mean, np.sqrt(np.sum(q * (values - mean) ** 2))

## 1. 宇宙クロノメーター: 銀河の年齢差から $H(z)$ を直接測る

スケール因子 $a = 1/(1+z)$ を時間で微分すると

$$
H(z) = \frac{\dot a}{a} = -\frac{1}{1+z}\frac{dz}{dt}
$$

なので、**赤方偏移が少しだけ違う 2 つの時代の、宇宙の年齢の差 $dt$** が分かれば、宇宙モデルも距離も使わずに $H(z)$ が求まります(Jimenez & Loeb 2002)。

時計として使うのは、ずっと昔に星をつくり終えて、そのあと静かに年をとっているだけの大質量の楕円銀河です。
スペクトルの特徴(4000 Å のブレイク $D4000$ の強さ)が年齢とともに単調に強くなるので、それを恒星進化のモデルで年齢差に換算します。

系統誤差としては、恒星進化のモデル(SPS)や初期質量関数(IMF)の違いが大きく、これはすべての赤方偏移に共通にかかるので、Moresco et al. (2020) にならって相関のある共分散行列にします。

In [ ]:
cc = pd.read_csv(CC_URL + "HzTable_MM_BC03.dat", comment="#", header=None,
                 names=["z", "H", "err", "err_stat", "err_met", "ref"])
# 赤方偏移ごとの系統誤差の割合 [%]
cc_sys = pd.read_csv(CC_URL + "data_MM20.dat", sep=r"\s+", comment="#", header=None,
                     names=["z", "imf", "stlib", "sps", "sps_ooo"])

z_cc = cc["z"].to_numpy()
H_cc = cc["H"].to_numpy()
cov_cc = np.diag(cc["err"].to_numpy() ** 2)  # 統計誤差 + 金属量(赤方偏移ごとに独立)
for col in ["sps_ooo", "imf"]:  # BC03 モデルを使った測定に対する、Moresco らの例と同じ組み合わせ
    frac = np.interp(z_cc, cc_sys["z"], cc_sys[col]) / 100
    cov_cc += np.outer(frac * H_cc, frac * H_cc)  # すべての z で完全に相関
cov_cc_inv = np.linalg.inv(cov_cc)
err_cc = np.sqrt(np.diag(cov_cc))
cc[["z", "H", "err", "ref"]]

平坦な ΛCDM の $H(z) = H_0 E(z)$ を当てはめて、$H_0$ と $\Omega_m$ を求めます。

In [ ]:
h0_ax = np.linspace(40, 100, 301)
om_ax = np.linspace(0.01, 1.0, 199)
H0G, OMG = np.meshgrid(h0_ax, om_ax)
resid = H_cc - H0G[..., None] * e_lcdm(z_cc, OMG[..., None])
c2_cc = np.einsum("...i,ij,...j->...", resid, cov_cc_inv, resid)
p_cc = np.exp(-(c2_cc - c2_cc.min()) / 2)
H0_cc, H0_cc_err = marginal(h0_ax, p_cc, axis=0)
om_cc, om_cc_err = marginal(om_ax, p_cc, axis=1)
print(f"χ²_min = {c2_cc.min():.1f}(データ {len(z_cc)} 点)")
print(f"宇宙クロノメーター: H0 = {H0_cc:.1f} ± {H0_cc_err:.1f} km/s/Mpc, Ωm = {om_cc:.2f} ± {om_cc_err:.2f}")

zz = np.linspace(0, 2.4, 200)
plt.figure(figsize=(7, 5))
plt.errorbar(z_cc, H_cc, yerr=err_cc, fmt="o", c="tab:green", label="cosmic chronometers")
plt.plot(zz, H0_cc * e_lcdm(zz, om_cc), c="tab:green", label=f"flat ΛCDM fit (H0 = {H0_cc:.1f})")
plt.xlabel("redshift z")
plt.ylabel("H(z) [km/s/Mpc]")
plt.legend()
plt.show()

誤差は大きいですが、距離はしごも CMB も使わずに膨張の速さが分かるのがこの方法の強みです(Moresco et al. 2020 の値は $H_0 = 66.5 \pm 5.4$)。
$z \approx 2$ で $H$ が今の 3 倍近くになっていて、昔ほど(割合として)速く膨張していたことが分かります。

## 2. BAO: 宇宙に刻まれた物差し

宇宙が生まれて約 38 万年までは、光子とバリオン(普通の物質)が一体になった流体の中を音波が伝わっていました。
光子が離れたとき、音波が進んだ距離 **$r_d \approx 147$ Mpc**(今の大きさに換算)が、物質の分布の「ここから $r_d$ 離れたところに銀河がやや多い」という特徴として残りました。これが **バリオン音響振動(BAO)** です。

銀河の 3 次元分布でこの物差しの見かけの大きさを測ると

- 視線方向の長さ: $\Delta z = r_d\, H(z) / c$ → $D_H / r_d$ が分かる($D_H \equiv c/H(z)$)
- 視線に垂直な方向の角度: $\Delta\theta = r_d / D_M(z)$ → $D_M / r_d$ が分かる($D_M$: 共動距離)
- 両方を平均したもの: $D_V / r_d$、$D_V = \left[z D_M^2 D_H\right]^{1/3}$(銀河の数が少ない最初の赤方偏移の測定)

です。どれも $r_d$ との **比** なので、$r_d$ が分からないと $H(z)$ の絶対値は決まりません。
ここでは、$r_d$ を Planck の CMB の観測から計算された値($147.09 \pm 0.26$ Mpc)にします。

In [ ]:
bao = pd.read_csv(BAO_URL + "desi_gaussian_bao_ALL_GCcomb_mean.txt", sep=r"\s+", comment="#", header=None,
                  names=["z", "value", "quantity"])
cov_bao = np.loadtxt(BAO_URL + "desi_gaussian_bao_ALL_GCcomb_cov.txt")
cov_bao_inv = np.linalg.inv(cov_bao)
bao["err"] = np.sqrt(np.diag(cov_bao))
bao

$D_H / r_d$ の測定は、そのまま $H(z) = c / (r_d \cdot D_H/r_d)$ に直せます。

In [ ]:
dh = bao[bao["quantity"] == "DH_over_rs"]
z_bao = dh["z"].to_numpy()
H_bao = C_KMS / (RD_PLANCK * dh["value"].to_numpy())
H_bao_err = H_bao * dh["err"].to_numpy() / dh["value"].to_numpy()
for z, h, e in zip(z_bao, H_bao, H_bao_err):
    print(f"z = {z:.3f}: H = {h:6.1f} ± {e:4.1f} km/s/Mpc")

すべての測定($D_M$, $D_V$ も含む)を平坦 ΛCDM に当てはめます。
予言値はどれも $\dfrac{c}{H_0 r_d} \times (\Omega_m \text{ だけで決まる関数})$ の形なので、
データから決まるのは $\Omega_m$ と $H_0 r_d$ の 2 つです。
振幅 $\alpha = c/(H_0 r_d)$ は 1 章の超新星の $\mathcal{A}$ と同じように解析的に最適化できます。

In [ ]:
Z_GRID = np.linspace(0, 2.5, 5001)


def bao_shape(om):
    # H0 × (DM, DH, DV) / c を、BAO の各測定について返す
    e = e_lcdm(Z_GRID, om)
    dc = np.concatenate([[0], np.cumsum((1 / e[1:] + 1 / e[:-1]) / 2 * np.diff(Z_GRID))])
    z = bao["z"].to_numpy()
    dm = np.interp(z, Z_GRID, dc)
    dh_ = 1 / e_lcdm(z, om)
    dv = (z * dm ** 2 * dh_) ** (1 / 3)
    q = bao["quantity"].to_numpy()
    return np.select([q == "DM_over_rs", q == "DH_over_rs", q == "DV_over_rs"], [dm, dh_, dv])


y_bao = bao["value"].to_numpy()
om_ax_bao = np.linspace(0.2, 0.45, 501)
c2_bao_1d, alpha_bao = [], []
for om in om_ax_bao:
    f = bao_shape(om)
    alpha = (f @ cov_bao_inv @ y_bao) / (f @ cov_bao_inv @ f)
    r = y_bao - alpha * f
    c2_bao_1d.append(r @ cov_bao_inv @ r)
    alpha_bao.append(alpha)
c2_bao_1d, alpha_bao = np.array(c2_bao_1d), np.array(alpha_bao)
k = np.argmin(c2_bao_1d)
om_bao = om_ax_bao[k]
h0rd_bao = C_KMS / alpha_bao[k]  # [km/s]
ok1 = c2_bao_1d - c2_bao_1d[k] < 1
print(f"χ²_min = {c2_bao_1d[k]:.1f}(データ {len(y_bao)} 点)")
print(f"DESI DR2 BAO: Ωm = {om_bao:.4f} (1σ: {om_ax_bao[ok1].min():.4f} 〜 {om_ax_bao[ok1].max():.4f})")
print(f"             H0 rd = {h0rd_bao / 100:.2f} × 100 km/s")
H0_bao = h0rd_bao / RD_PLANCK
print(f"r_d = {RD_PLANCK} Mpc とすると H0 = {H0_bao:.1f} km/s/Mpc")

DESI DR2 の論文の値は、BAO だけで $\Omega_m = 0.2975 \pm 0.0086$、$H_0 r_d = (101.54 \pm 0.73) \times 100$ km/s です。
$r_d$ に CMB の値を使うと $H_0 \approx 69$ になり、超新星+セファイドの 73 よりかなり小さくなります。

> BAO の $H_0$ は、$r_d$ をどう決めるかに左右されます。$r_d$ は初期宇宙の物理(バリオン密度や、光子・ニュートリノの量)で決まるので、
> 「$r_d$ が CMB からの計算より 7% ほど小さければ両方と合う」というのが、ハッブル・テンションを解く案の 1 つ(初期ダークエネルギーなど)になっています。

## 3. 超新星: モデルを仮定せずに $H(z)/H_0$ を出す

超新星の光度距離は、$H(z)$ の逆数を積分したものです(平坦な宇宙を仮定)。

$$
\frac{H_0\, d_L}{c} = (1+z_{\mathrm{hel}}) \int_0^{z} \frac{dz'}{E(z')}
$$

[sn_cosmology_fit.ipynb](sn_cosmology_fit.ipynb) では $E(z)$ に ΛCDM の式を使いましたが、ここでは **いくつかの赤方偏移 $z_k$ での値 $E_k$ を自由なパラメータ** にして、その間は直線でつなぎます($E(0) = 1$ は定義)。
こうすると、ΛCDM を仮定せずに膨張の歴史の形が分かります。

超新星の数が少ない $z > 0.8$ は不安定になるので使いません。

In [ ]:
raw = pd.read_csv(SN_URL + "Pantheon%2BSH0ES.dat", sep=r"\s+")
cov_flat = pd.read_csv(SN_URL + "Pantheon%2BSH0ES_STAT%2BSYS.cov", header=None)[0].to_numpy()
n_all = int(cov_flat[0])
cov_sn_all = cov_flat[1:].reshape(n_all, n_all)

Z_MAX_SN = 0.8
use = ((raw["zHD"] > 0.01) & (raw["zHD"] < Z_MAX_SN) & (raw["IS_CALIBRATOR"] == 0)).to_numpy()
sn = raw[use]
m_sn = sn["m_b_corr"].to_numpy()
z_hd, z_hel = sn["zHD"].to_numpy(), sn["zHEL"].to_numpy()
# コレスキー分解 C = L L^T を使い、残差に L^{-1} をかけると、χ² が単純な 2 乗和になる
L_inv = np.linalg.inv(np.linalg.cholesky(cov_sn_all[np.ix_(use, use)]))
print(f"使う超新星の行: {len(sn)}")

In [ ]:
NODES = np.array([0.0, 0.15, 0.3, 0.5, 0.8])
Z_SN_GRID = np.linspace(0, Z_MAX_SN, 1601)


def m_model(params):
    # params = [A, E(z_1), E(z_2), ...]
    A, e_nodes = params[0], np.concatenate([[1.0], params[1:]])
    inv_e = 1 / np.interp(Z_SN_GRID, NODES, e_nodes)
    dc = np.concatenate([[0], np.cumsum((inv_e[1:] + inv_e[:-1]) / 2 * np.diff(Z_SN_GRID))])
    return A + 5 * np.log10(np.interp(z_hd, Z_SN_GRID, dc) * (1 + z_hel))


def whitened_residual(params):
    return L_inv @ (m_sn - m_model(params))


p0 = np.concatenate([[24.0], e_lcdm(NODES[1:], 0.3)])
fit = least_squares(whitened_residual, p0)
cov_params = np.linalg.inv(fit.jac.T @ fit.jac)
E_sn = fit.x[1:]
E_sn_err = np.sqrt(np.diag(cov_params))[1:]
print(f"χ²_min = {2 * fit.cost:.1f}")
for z, e, s in zip(NODES[1:], E_sn, E_sn_err):
    print(f"z = {z:.2f}: H/H0 = {e:.3f} ± {s:.3f}   (平坦 ΛCDM Ωm=0.33 なら {e_lcdm(z, 0.33):.3f})")

corr = cov_params[1:, 1:] / np.outer(E_sn_err, E_sn_err)
print("隣り合う点の相関係数:", np.round(np.diag(corr, 1), 2))

隣り合う点どうしには負の相関があります(片方が高めに出るともう片方が低めに出て、積分した距離はつじつまが合う)。
グラフでは誤差棒だけを描きますが、点がジグザグしやすいことは頭に入れておきます。

$H(z)$ の絶対値にするには $H_0$ をかけます。ここでは超新星+セファイドの値(73.4)を使います。

In [ ]:
H_sn = H0_SH0ES * E_sn
H_sn_err = H_sn * np.hypot(E_sn_err / E_sn, H0_SH0ES_ERR / H0_SH0ES)

## 4. 3 つを重ねる

### (a) $H(z)$ の絶対値

比較のため、Planck(CMB)の ΛCDM と、超新星+セファイドの $H_0$ を使った ΛCDM の曲線も描きます。

In [ ]:
zz = np.linspace(0, 2.4, 300)
z0 = np.array([0.0])

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(zz, PLANCK["H0"] * e_lcdm(zz, PLANCK["Om"]), c="k", lw=1, label="Planck ΛCDM (H0=67.4, Ωm=0.315)")
ax.plot(zz, H0_SH0ES * e_lcdm(zz, 0.333), c="tab:red", lw=1, ls="--", label=f"SN-fit ΛCDM (H0={H0_SH0ES}, Ωm=0.333)")
ax.errorbar(z_cc, H_cc, yerr=err_cc, fmt="o", c="tab:green", label="cosmic chronometers")
ax.errorbar(z_bao, H_bao, yerr=H_bao_err, fmt="s", c="tab:blue", label=f"DESI DR2 BAO (rd = {RD_PLANCK} Mpc)")
ax.errorbar(NODES[1:], H_sn, yerr=H_sn_err, fmt="D", c="tab:red", label="Pantheon+ SNe × SH0ES H0")
ax.errorbar(z0, [H0_SH0ES], yerr=[H0_SH0ES_ERR], fmt="*", ms=12, c="tab:red")
ax.errorbar(z0 + 0.03, [PLANCK["H0"]], yerr=[PLANCK["H0_err"]], fmt="*", ms=12, c="k")
ax.set_xlabel("redshift z")
ax.set_ylabel("H(z) [km/s/Mpc]")
ax.legend(loc="upper left")
plt.show()

$z \lesssim 1$ を拡大して、$\dot{a} = H(z)/(1+z)$ にすると違いが見やすくなります(右肩下がりの部分が加速膨張)。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
zz = np.linspace(0, 2.4, 300)
ax.plot(zz, PLANCK["H0"] * e_lcdm(zz, PLANCK["Om"]) / (1 + zz), c="k", lw=1, label="Planck ΛCDM")
ax.plot(zz, H0_SH0ES * e_lcdm(zz, 0.333) / (1 + zz), c="tab:red", lw=1, ls="--", label="SN-fit ΛCDM")
ax.errorbar(z_cc, H_cc / (1 + z_cc), yerr=err_cc / (1 + z_cc), fmt="o", c="tab:green", label="cosmic chronometers")
ax.errorbar(z_bao, H_bao / (1 + z_bao), yerr=H_bao_err / (1 + z_bao), fmt="s", c="tab:blue", label="DESI DR2 BAO")
ax.errorbar(NODES[1:], H_sn / (1 + NODES[1:]), yerr=H_sn_err / (1 + NODES[1:]), fmt="D", c="tab:red", label="Pantheon+ SNe")
ax.errorbar(z0, [H0_SH0ES], yerr=[H0_SH0ES_ERR], fmt="*", ms=12, c="tab:red")
ax.errorbar(z0 + 0.03, [PLANCK["H0"]], yerr=[PLANCK["H0_err"]], fmt="*", ms=12, c="k")
ax.set_xlabel("redshift z")
ax.set_ylabel("H(z)/(1+z) = da/dt [km/s/Mpc]")
ax.set_ylim(40, 110)
ax.legend()
plt.show()

超新星の点(と $z = 0$ の赤い星)は Planck の曲線より全体に上にあり、BAO の点は Planck の曲線にぴったり乗ります。
宇宙クロノメーターは誤差が大きく、どちらとも矛盾しません。

### (b) 形だけを比べる: $H(z)/H_0$

それぞれの方法で決めた $H_0$(BAO は $H_0 r_d$)で割って、**膨張の歴史の形** だけを比べます。

In [ ]:
E_cc = H_cc / H0_cc
E_cc_err = err_cc / H0_cc
E_bao = (C_KMS / h0rd_bao) / dh["value"].to_numpy()  # (c / (H0 rd)) / (DH/rd) = H/H0
E_bao_err = E_bao * dh["err"].to_numpy() / dh["value"].to_numpy()

zz = np.linspace(0, 2.4, 300)
fig, ax = plt.subplots(figsize=(8, 5))
for om, ls in [(0.3, "-"), (1.0, "--")]:
    ax.plot(zz, e_lcdm(zz, om), c="gray", ls=ls, lw=1, label=f"flat, Ωm = {om}")
ax.errorbar(z_cc, E_cc, yerr=E_cc_err, fmt="o", c="tab:green", label="cosmic chronometers (÷ own H0)")
ax.errorbar(z_bao, E_bao, yerr=E_bao_err, fmt="s", c="tab:blue", label="DESI DR2 BAO (÷ own H0 rd)")
ax.errorbar(NODES[1:], E_sn, yerr=E_sn_err, fmt="D", c="tab:red", label="Pantheon+ SNe")
ax.set_xlabel("redshift z")
ax.set_ylabel("H(z) / H0")
ax.set_ylim(0.8, 5)
ax.legend(loc="upper left")
plt.show()

形だけを比べると、3 つの方法はよく合っていて、どれも「物質だけの宇宙」(破線)ではなく、$\Omega_m \approx 0.3$ のダークエネルギー入りの宇宙を示しています。

つまり、**膨張の歴史の形はそろっているのに、目盛りの合わせ方(距離はしごか、CMB で決まる $r_d$ か)で絶対値が 8% ほど食い違う**、というのがハッブル・テンションの姿です。

## 5. $\Omega_m$–$H_0$ 平面で比べる

3 つの方法の結果を、平坦 ΛCDM のパラメータ平面に並べます。

- 宇宙クロノメーター: 1 章の χ²
- BAO: $r_d = 147.09$ Mpc として、$\Omega_m$ と $H_0$ の χ²
- 超新星: 形から決まる $\Omega_m$(平坦 ΛCDM を当てはめる)と、セファイドで決まる $H_0$ の組み合わせ
- Planck: 論文の値(1σ の楕円で近似)

In [ ]:
# BAO: (Ωm, H0) の格子で χ² を計算(r_d は Planck の値に固定)
h0_ax2 = np.linspace(55, 85, 301)
om_ax2 = np.linspace(0.1, 0.6, 251)
H0G2, OMG2 = np.meshgrid(h0_ax2, om_ax2)
c2_bao = np.empty(H0G2.shape)
for i, om in enumerate(om_ax2):
    f = bao_shape(om)
    pred = (C_KMS / (h0_ax2[:, None] * RD_PLANCK)) * f[None, :]
    r = y_bao[None, :] - pred
    c2_bao[i] = np.einsum("ij,jk,ik->i", r, cov_bao_inv, r)

# 宇宙クロノメーター: 同じ格子で計算し直す
resid = H_cc - H0G2[..., None] * e_lcdm(z_cc, OMG2[..., None])
c2_cc2 = np.einsum("...i,ij,...j->...", resid, cov_cc_inv, resid)

# 超新星(z < 0.8 だけ使っていることに注意): 平坦 ΛCDM の Ωm の χ² + SH0ES の H0
one_w = L_inv @ np.ones(len(m_sn))


def c2_sn_flat(om):
    e = e_lcdm(Z_SN_GRID, om)
    dc = np.concatenate([[0], np.cumsum((1 / e[1:] + 1 / e[:-1]) / 2 * np.diff(Z_SN_GRID))])
    r_w = L_inv @ (m_sn - 5 * np.log10(np.interp(z_hd, Z_SN_GRID, dc) * (1 + z_hel)))
    return r_w @ r_w - (one_w @ r_w) ** 2 / (one_w @ one_w)


c2_sn_om = np.array([c2_sn_flat(om) for om in om_ax2])
c2_sn = (c2_sn_om - c2_sn_om.min())[:, None] + ((h0_ax2[None, :] - H0_SH0ES) / H0_SH0ES_ERR) ** 2

c2_planck = ((H0G2 - PLANCK["H0"]) / PLANCK["H0_err"]) ** 2 + ((OMG2 - PLANCK["Om"]) / PLANCK["Om_err"]) ** 2

fig, ax = plt.subplots(figsize=(7, 6))
for c2, color, label in [
    (c2_cc2, "tab:green", "cosmic chronometers"),
    (c2_bao, "tab:blue", "DESI DR2 BAO + rd(Planck)"),
    (c2_sn, "tab:red", "Pantheon+ (z<0.8) + SH0ES H0"),
    (c2_planck, "k", "Planck 2018 CMB"),
]:
    d = c2 - c2.min()
    ax.contourf(OMG2, H0G2, d, levels=[0, LEVELS_2D[0]], colors=[color], alpha=0.5)
    ax.contourf(OMG2, H0G2, d, levels=LEVELS_2D, colors=[color], alpha=0.2)
    ax.contour(OMG2, H0G2, d, levels=LEVELS_2D, colors=color, linewidths=0.8)
    ax.plot([], [], c=color, lw=6, alpha=0.5, label=label)
ax.set_xlabel("Ωm")
ax.set_ylabel("H0 [km/s/Mpc]")
ax.set_xlim(0.1, 0.6)
ax.set_ylim(55, 85)
ax.set_title("flat ΛCDM: 1σ, 2σ")
ax.legend(loc="upper right")
plt.show()

om_sn = om_ax2[np.argmin(c2_sn_om)]
print(f"超新星(z < 0.8)だけの平坦 ΛCDM: Ωm = {om_sn:.3f}")

- **$\Omega_m$(横方向)** は、超新星・BAO・CMB で 0.30〜0.35 とおおむねそろっています(超新星は $z < 0.8$ だけなので 0.35。全部使うと 0.33)
- **$H_0$(縦方向)** は、BAO+CMB の物差しを使うグループ(67〜69)と、セファイドの距離はしごを使う超新星(73)に分かれます
- 宇宙クロノメーターは誤差が大きく、今のところ両方を含んでしまいます。誤差が半分以下になれば、どちら側か判定できるはずです

BAO と超新星で $\Omega_m$ が少しずれている(0.30 と 0.33)のも、DESI が「ダークエネルギーが時間変化している($w_0 w_a$ モデル)ほうがよく合う」と報告している理由の 1 つです。

## まとめ

実行すると、だいたい次のようになります。

| 方法 | $H_0$ [km/s/Mpc] | $\Omega_m$(平坦 ΛCDM) | 物差し |
| --- | --- | --- | --- |
| 宇宙クロノメーター | $66 \pm 6$ | $0.37 \pm 0.11$ | 恒星進化モデル |
| DESI DR2 BAO | 69.0($r_d = 147.09$ Mpc のとき) | $0.298 \pm 0.009$ | CMB で決まる $r_d$ |
| Pantheon+ 超新星 | 73.4 ± 1.0(セファイド) | 0.35($z < 0.8$)/ $0.333 \pm 0.018$(全部) | 距離はしご |
| Planck CMB | $67.4 \pm 0.5$ | $0.315 \pm 0.007$ | (ΛCDM を仮定) |

## 発展メモ

- 宇宙クロノメーターの系統誤差の共分散を外す(対角だけにする)と、$H_0$ の誤差はどれだけ小さくなるか
- 超新星の節点 `NODES` の位置や数を変えると、$H(z)/H_0$ はどう変わるか
- BAO の $r_d$ を自由にして、超新星+SH0ES の $H_0$ と組み合わせると $r_d$ はいくつになるか(「距離はしごで決めた $r_d$」)
- BAO と超新星を合わせて、$w(a) = w_0 + w_a(1-a)$ を当てはめ、DESI の「時間変化するダークエネルギー」の兆候を自分で確かめる

## 参考・クレジット

- Jimenez & Loeb (2002), *Constraining Cosmological Parameters Based on Relative Galaxy Ages*, ApJ 573, 37
- Moresco et al. (2012), *Improved constraints on the expansion rate of the Universe up to z ~ 1.1 from the spectroscopic evolution of cosmic chronometers*, JCAP 08, 006
- Moresco (2015), *Raising the bar: new constraints on the Hubble parameter with cosmic chronometers at z ~ 2*, MNRAS 450, L16
- Moresco et al. (2016), *A 6% measurement of the Hubble parameter at z ~ 0.45: direct evidence of the epoch of cosmic re-acceleration*, JCAP 05, 014
- Moresco et al. (2020), *Setting the Stage for Cosmic Chronometers. II. Impact of Stellar Population Synthesis Models Systematics and Full Covariance Matrix*, ApJ 898, 82
- DESI Collaboration (2025), *DESI DR2 Results II: Measurements of Baryon Acoustic Oscillations and Cosmological Constraints*, arXiv:2503.14738
- Scolnic et al. (2022), ApJ 938, 113 / Brout et al. (2022), ApJ 938, 110(Pantheon+)
- Riess et al. (2022), ApJL 934, L7(SH0ES)
- Planck Collaboration (2020), *Planck 2018 results. VI. Cosmological parameters*, A&A 641, A6